# Twitch Gamers Dataset - Comprehensive Data Exploration

This notebook performs a comprehensive exploration of the Twitch Gamers dataset including:
- Loading and inspecting the data
- Descriptive statistics for all features
- Data quality checks (missing values, outliers)
- Distribution visualizations
- Network structure analysis
- Feature engineering (network metrics)

**Dataset:** Large Twitch social network
- 168,114 nodes (Twitch users)
- 6,797,557 edges (mutual follower relationships)
- 9 node attributes

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Load the Dataset

In [ ]:
# Define data paths
DATA_PATH = '../data/'
EDGES_FILE = DATA_PATH + 'large_twitch_edges.csv'
FEATURES_FILE = DATA_PATH + 'large_twitch_features.csv'

# Load edges
print("Loading edges...")
edges_df = pd.read_csv(EDGES_FILE)
print(f"Edges loaded: {len(edges_df):,} rows")
print(f"Shape: {edges_df.shape}")
print(f"Columns: {edges_df.columns.tolist()}")
print("\nFirst few rows:")
print(edges_df.head())

In [ ]:
# Load features
print("Loading features...")
features_df = pd.read_csv(FEATURES_FILE)
print(f"Features loaded: {len(features_df):,} rows")
print(f"Shape: {features_df.shape}")
print(f"Columns: {features_df.columns.tolist()}")
print("\nFirst few rows:")
print(features_df.head())

## 2. Data Quality Checks

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing = features_df.isnull().sum()
missing_pct = (missing / len(features_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])
if missing_df['Missing Count'].sum() == 0:
    print("No missing values found!")

In [ ]:
# Check for duplicate nodes
print("=== Duplicate Check ===")
duplicates = features_df.duplicated(subset=['new_id']).sum()
print(f"Duplicate node IDs: {duplicates}")

# Check data types
print("\n=== Data Types ===")
print(features_df.dtypes)

## 3. Descriptive Statistics

In [ ]:
# Overall statistics
print("=== Descriptive Statistics ===")
print(features_df.describe())

In [ ]:
# Categorical features distribution
print("=== Binary Features Distribution ===")
binary_cols = ['mature', 'dead_account', 'affiliate']

for col in binary_cols:
    if col in features_df.columns:
        print(f"\n{col.upper()}:")
        counts = features_df[col].value_counts()
        percentages = features_df[col].value_counts(normalize=True) * 100
        result = pd.DataFrame({
            'Count': counts,
            'Percentage': percentages
        })
        print(result)

In [ ]:
# Language distribution (multi-class)
print("=== Language Distribution (Top 20) ===")
if 'language' in features_df.columns:
    lang_counts = features_df['language'].value_counts().head(20)
    lang_pct = (lang_counts / len(features_df)) * 100
    lang_df = pd.DataFrame({
        'Count': lang_counts,
        'Percentage': lang_pct
    })
    print(lang_df)
    print(f"\nTotal unique languages: {features_df['language'].nunique()}")

In [ ]:
# Numerical features statistics
print("=== Numerical Features Statistics ===")
numerical_cols = ['views', 'life_time', 'created_at', 'updated_at']

for col in numerical_cols:
    if col in features_df.columns:
        print(f"\n{col.upper()}:")
        print(f"  Mean: {features_df[col].mean():,.2f}")
        print(f"  Median: {features_df[col].median():,.2f}")
        print(f"  Std: {features_df[col].std():,.2f}")
        print(f"  Min: {features_df[col].min():,.2f}")
        print(f"  Max: {features_df[col].max():,.2f}")
        print(f"  Q1 (25%): {features_df[col].quantile(0.25):,.2f}")
        print(f"  Q3 (75%): {features_df[col].quantile(0.75):,.2f}")

## 4. Data Visualizations

In [ ]:
# Binary features distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
binary_cols = ['mature', 'dead_account', 'affiliate']

for idx, col in enumerate(binary_cols):
    if col in features_df.columns:
        features_df[col].value_counts().plot(kind='bar', ax=axes[idx], color=['#1f77b4', '#ff7f0e'])
        axes[idx].set_title(f'{col.capitalize()} Distribution', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(col.capitalize())
        axes[idx].set_ylabel('Count')
        axes[idx].tick_params(axis='x', rotation=0)
        
        # Add count labels on bars
        for container in axes[idx].containers:
            axes[idx].bar_label(container, fmt='%d')

plt.tight_layout()
plt.savefig('../results/binary_features_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Top 15 languages
if 'language' in features_df.columns:
    plt.figure(figsize=(12, 6))
    top_langs = features_df['language'].value_counts().head(15)
    top_langs.plot(kind='barh', color='steelblue')
    plt.title('Top 15 Languages on Twitch', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Users')
    plt.ylabel('Language')
    plt.gca().invert_yaxis()
    
    # Add count labels
    for i, v in enumerate(top_langs.values):
        plt.text(v + 500, i, f'{v:,}', va='center')
    
    plt.tight_layout()
    plt.savefig('../results/language_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Views distribution (log scale due to skewness)
if 'views' in features_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Regular scale
    axes[0].hist(features_df['views'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_title('Views Distribution (Linear Scale)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Views')
    axes[0].set_ylabel('Frequency')
    
    # Log scale
    axes[1].hist(np.log10(features_df['views'] + 1), bins=50, color='coral', edgecolor='black', alpha=0.7)
    axes[1].set_title('Views Distribution (Log10 Scale)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Log10(Views + 1)')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig('../results/views_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Life time distribution
if 'life_time' in features_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Regular scale
    axes[0].hist(features_df['life_time'], bins=50, color='green', edgecolor='black', alpha=0.7)
    axes[0].set_title('Life Time Distribution (Linear Scale)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Life Time (days)')
    axes[0].set_ylabel('Frequency')
    
    # Log scale
    axes[1].hist(np.log10(features_df['life_time'] + 1), bins=50, color='purple', edgecolor='black', alpha=0.7)
    axes[1].set_title('Life Time Distribution (Log10 Scale)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Log10(Life Time + 1)')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig('../results/lifetime_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation heatmap for numerical features
numerical_features = features_df.select_dtypes(include=[np.number])
if len(numerical_features.columns) > 1:
    plt.figure(figsize=(10, 8))
    correlation_matrix = numerical_features.corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Network Structure Analysis

In [ ]:
# Build the network graph
print("Building network graph...")
print("This may take a few minutes for a large graph...")

G = nx.Graph()

# Add nodes
G.add_nodes_from(features_df['new_id'].values)
print(f"Nodes added: {G.number_of_nodes():,}")

# Add edges
edge_list = [(row['numeric_id_1'], row['numeric_id_2']) for _, row in edges_df.iterrows()]
G.add_edges_from(edge_list)
print(f"Edges added: {G.number_of_edges():,}")

print("\nNetwork graph built successfully!")

In [ ]:
# Basic network statistics
print("=== Network Statistics ===")
print(f"Number of nodes: {G.number_of_nodes():,}")
print(f"Number of edges: {G.number_of_edges():,}")
print(f"Network density: {nx.density(G):.6f}")
print(f"Is connected: {nx.is_connected(G)}")

# Connected components
num_components = nx.number_connected_components(G)
print(f"Number of connected components: {num_components}")

if num_components > 1:
    components = list(nx.connected_components(G))
    component_sizes = [len(c) for c in components]
    print(f"Largest component size: {max(component_sizes):,} nodes")
    print(f"Smallest component size: {min(component_sizes)} nodes")

## 6. Network Feature Engineering

In [ ]:
# Calculate degree centrality
print("Calculating degree centrality...")
degree_dict = dict(G.degree())
features_df['degree'] = features_df['new_id'].map(degree_dict)

print("\nDegree statistics:")
print(features_df['degree'].describe())

In [ ]:
# Degree distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(features_df['degree'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.title('Degree Distribution (Linear Scale)', fontsize=12, fontweight='bold')
plt.xlabel('Degree')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(np.log10(features_df['degree'] + 1), bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.title('Degree Distribution (Log10 Scale)', fontsize=12, fontweight='bold')
plt.xlabel('Log10(Degree + 1)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig('../results/degree_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate clustering coefficient (for a sample if graph is too large)
print("Calculating clustering coefficient...")
print("Note: This may take a while for large graphs.")
print("Computing for all nodes...")

# For very large graphs, you might want to sample
# sample_nodes = np.random.choice(list(G.nodes()), size=min(10000, len(G)), replace=False)
# clustering_dict = {node: nx.clustering(G, node) for node in sample_nodes}

clustering_dict = nx.clustering(G)
features_df['clustering_coefficient'] = features_df['new_id'].map(clustering_dict)

print("\nClustering coefficient statistics:")
print(features_df['clustering_coefficient'].describe())

In [ ]:
# Clustering coefficient distribution
plt.figure(figsize=(10, 5))
plt.hist(features_df['clustering_coefficient'].dropna(), bins=50, color='green', edgecolor='black', alpha=0.7)
plt.title('Clustering Coefficient Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Clustering Coefficient')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/clustering_coefficient_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate PageRank (this can be slow for large graphs)
print("Calculating PageRank...")
print("This may take several minutes for large graphs...")

pagerank_dict = nx.pagerank(G, max_iter=100)
features_df['pagerank'] = features_df['new_id'].map(pagerank_dict)

print("\nPageRank statistics:")
print(features_df['pagerank'].describe())

In [ ]:
# PageRank distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(features_df['pagerank'], bins=50, color='purple', edgecolor='black', alpha=0.7)
plt.title('PageRank Distribution (Linear Scale)', fontsize=12, fontweight='bold')
plt.xlabel('PageRank')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(np.log10(features_df['pagerank']), bins=50, color='orange', edgecolor='black', alpha=0.7)
plt.title('PageRank Distribution (Log10 Scale)', fontsize=12, fontweight='bold')
plt.xlabel('Log10(PageRank)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig('../results/pagerank_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Save Enhanced Dataset

In [ ]:
# Save the enhanced dataset with network features
output_file = '../data/enhanced_features.csv'
features_df.to_csv(output_file, index=False)
print(f"Enhanced dataset saved to: {output_file}")
print(f"New features added: degree, clustering_coefficient, pagerank")
print(f"Total features: {len(features_df.columns)}")

## 8. Summary and Key Insights

In [ ]:
print("=== DATA EXPLORATION SUMMARY ===")
print(f"\nDataset Size: {len(features_df):,} nodes, {len(edges_df):,} edges")
print(f"\nFeatures: {len(features_df.columns)} total")
print(f"  - Original features: {len(features_df.columns) - 3}")
print(f"  - Engineered network features: 3 (degree, clustering_coefficient, pagerank)")

print("\n=== Key Insights ===")
print(f"\n1. Binary Classification Targets:")
for col in ['affiliate', 'mature', 'dead_account']:
    if col in features_df.columns:
        class_dist = features_df[col].value_counts(normalize=True) * 100
        print(f"   {col}: {class_dist.to_dict()}")

print(f"\n2. Multi-class Classification Target:")
if 'language' in features_df.columns:
    print(f"   Languages: {features_df['language'].nunique()} unique classes")
    print(f"   Most common: {features_df['language'].mode()[0]} ({(features_df['language'].value_counts().iloc[0] / len(features_df) * 100):.1f}%)")

print(f"\n3. Regression Targets:")
if 'views' in features_df.columns:
    print(f"   Views: Mean={features_df['views'].mean():,.0f}, Median={features_df['views'].median():,.0f}, Max={features_df['views'].max():,.0f}")
if 'life_time' in features_df.columns:
    print(f"   Life Time: Mean={features_df['life_time'].mean():,.0f} days, Median={features_df['life_time'].median():,.0f} days")

print(f"\n4. Network Characteristics:")
print(f"   Average degree: {features_df['degree'].mean():.2f}")
print(f"   Average clustering coefficient: {features_df['clustering_coefficient'].mean():.4f}")
print(f"   Network density: {nx.density(G):.6f}")

print("\n=== Next Steps ===")
print("1. Data preprocessing: handle outliers, normalize/standardize features")
print("2. Feature selection: identify most important features for each task")
print("3. Train/test split: create proper evaluation setup")
print("4. Baseline models: implement simple baselines for comparison")
print("5. Advanced models: experiment with various algorithms")
print("6. Hyperparameter tuning: optimize model performance")
print("7. Error analysis: understand model failures")